# Mapping a 19th-Century Census: Cossacks of the Poltava Governorate (1850)
**Author:** Ivan Hromovyi

An end-to-end geospatial exploratory analysis built in Python (pandas · GeoPandas · Altair).

**Core question:** did the *Cossack* (Malorosiyski kozaky) population of the Poltava Governorate behave differently from the rest of the province - in land allocation and demographics - or were "Cossack" districts essentially like everywhere else?

## Questions

1. How is the Cossack population distributed across districts (*povits*) - absolute count vs share of population?
2. What is the ranking of districts from most to least "Cossack"?
3. Does a district's Cossack share affect land-per-capita (state vs landlord peasants)?
4. Do "Cossack" districts show different demographic dynamics (population reproduction) than the province as a whole?

## 1. Setup

In [ ]:
import pandas as pd
import geopandas as gpd
import altair as alt

# Tiny dataset - lift Altair's default 5,000-row cap just in case
alt.data_transformers.disable_max_rows()

In [ ]:
# --- Paths (adjust to your machine; forward slashes work on Windows too) ---
EXCEL_PATH   = 'C:/Users/Ivan/Documents/Code in University/Data manipulation essentials/final project/Народонаселення Полтавської губернії.xlsx'
GEOJSON_PATH = 'C:/Users/Ivan/OneDrive/Documents/Code in University/Data manipulation essentials/final project/final_data.txt'

# District name -> English label (charts use English; merge keys stay original)
DISTRICT_EN = {
    'Гадяцький': 'Hadiach', 'Зіньківський': 'Zinkiv', 'Золотоніський': 'Zolotonosha',
    'Кобеляцький': 'Kobeliaky', 'Костянтиноградський': 'Kostiantynohrad',
    'Кременчуцький': 'Kremenchuk', 'Лохвицький': 'Lokhvytsia', 'Лубенський': 'Lubny',
    'Миргородський': 'Myrhorod', 'Переяславський': 'Pereiaslav', 'Полтавський': 'Poltava',
    'Прилуцький': 'Pryluky', 'Пирятинський': 'Piryatyn', 'Роменський': 'Romny',
    'Хорольський': 'Khorol', 'Губернія загалом': 'Governorate total',
}

# Districts with the highest Cossack share (highlighted later)
COSSACK_DISTRICTS = ['Зіньківський', 'Миргородський', 'Кобеляцький', 'Лохвицький', 'Лубенський']

## 2. Load the data
The source is a multi-sheet historical Excel workbook plus a GeoJSON of district borders.

In [ ]:
# Population (1850), land-per-capita (1848), demographic reproduction
pop_1850  = pd.read_excel(EXCEL_PATH, sheet_name='1_Народонаселення на 1850 р.')
land_raw  = pd.read_excel(EXCEL_PATH, sheet_name='7_Земля на ревізьку душу 1848')
repro_raw = pd.read_excel(EXCEL_PATH, sheet_name='13_Відтворення населення')

# District borders
gdf = gpd.read_file(GEOJSON_PATH)

## 3. Clean the 1850 population sheet
The sheet interleaves male/female columns. We keep the male totals (marked `о.п.`), label the 15 districts, and pull two rows - Cossacks and total population - into a tidy table.

In [ ]:
pop = pop_1850.set_index(pop_1850.columns[0])

# keep only male-population columns ('о.п.') plus the label column
male_cols = (pop.iloc[0] == 'о.п.')
male_cols.loc['Станом на 1850 рік'] = True
pop = pop.loc[:, male_cols]

# label the districts (order matches the source sheet)
pop.columns = list(DISTRICT_EN.keys())

# reshape to one row per district
cossacks = pop.loc[['Малороссийских козаков', 'Итого']].T.reset_index()
cossacks.columns = ['district_ua', 'cossacks', 'total']
cossacks = cossacks.head(15)                       # drop the 'Governorate total' row
cossacks['district'] = cossacks['district_ua'].map(DISTRICT_EN)
cossacks['cossack_share'] = cossacks['cossacks'] / cossacks['total']
cossacks

## 4. Join the census to district geometry
Repair invalid polygons with `buffer(0)`, then merge the numbers onto the map.

In [ ]:
gdf['geometry'] = gdf['geometry'].buffer(0)        # fix geometry / winding order
geo = gdf.merge(cossacks, left_on='name', right_on='district_ua')
geo[['name', 'cossacks', 'total', 'cossack_share']].head()

## Q1 - Where do the Cossacks live? (choropleth maps)

In [ ]:
def choropleth(field, title, scheme='greens'):
    return alt.Chart(geo, title=title).mark_geoshape(stroke='white', strokeWidth=0.5).encode(
        color=alt.Color(f'{field}:Q', title=title, scale=alt.Scale(scheme=scheme)),
        tooltip=[alt.Tooltip('district:N', title='District'),
                 alt.Tooltip('cossacks:Q', title='Cossacks', format=','),
                 alt.Tooltip('total:Q', title='Total population', format=','),
                 alt.Tooltip('cossack_share:Q', title='Cossack share', format='.0%')]
    ).project(type='mercator').properties(width=300, height=260)

map_total = choropleth('total', 'Total population (1850)', 'blues')
map_count = choropleth('cossacks', 'Cossack population (count)', 'blues')
map_share = choropleth('cossack_share', 'Cossack share per capita', 'greens')

(map_total | map_count | map_share).resolve_scale(color='independent')

**Finding.** The largest Cossack *counts* sit in Kobeliaky, Zolotonosha, Zinkiv and Romny; the highest Cossack *share per capita* is in Zinkiv, Lokhvytsia and Lubny. These maps are descriptive - they tell us where to look before the analysis.

## Q2 - Ranking districts by Cossack share

In [ ]:
rank = alt.Chart(geo, title='Districts ranked by Cossack share').mark_bar().encode(
    x=alt.X('cossack_share:Q', title='Cossack share', axis=alt.Axis(format='%')),
    y=alt.Y('district:N', title=None, sort='-x'),
    color=alt.Color('cossack_share:Q', scale=alt.Scale(scheme='greens'), legend=None),
    tooltip=[alt.Tooltip('district:N', title='District'),
             alt.Tooltip('cossack_share:Q', title='Share', format='.0%')]
).properties(width=520, height=320)
rank

**Finding.** Zinkiv, Kobeliaky and Lubny top the share ranking - the candidates to test against land and demographics below.

## Q3 - Does Cossack share affect land-per-capita?
Join the land sheet, then look for a relationship - separately for state and landlord peasants.

In [ ]:
land = (land_raw.rename(columns={
            'Unnamed: 0': 'district_ua',
            'Землі на душу населення': 'land_landlord',   # landlord peasants
            'Unnamed: 8': 'land_state',                    # state peasants
            'Unnamed: 9': 'land_total'})
        [['district_ua', 'land_landlord', 'land_state', 'land_total']]
        .iloc[1:17])

land_geo = land.merge(cossacks, on='district_ua')
land_geo[['district', 'cossack_share', 'land_state', 'land_landlord']].head()

In [ ]:
def land_scatter(yfield, label):
    base = alt.Chart(land_geo, title=f'Cossack share vs land per capita ({label})').encode(
        x=alt.X('cossack_share:Q', title='Cossack share', axis=alt.Axis(format='%')),
        y=alt.Y(f'{yfield}:Q', title=f'Land per capita ({label})'))
    pts = base.mark_circle(size=90, opacity=0.7, color='#2e7d32').encode(
        tooltip=['district:N', alt.Tooltip('cossack_share:Q', format='.0%'), f'{yfield}:Q'])
    trend = base.transform_regression('cossack_share', yfield).mark_line(color='red', strokeDash=[4, 4])
    return (pts + trend).properties(width=360, height=300)

land_scatter('land_state', 'state peasants') | land_scatter('land_landlord', 'landlord peasants')

**Finding.** **No relationship.** The points scatter with no trend (flat regression line) - a district's Cossack share does not explain land-per-capita for either peasant group.

## Q4 - Different demographic dynamics? (reproduction over time)
Reshape the reproduction sheet **wide -> long**, then highlight the top-Cossack districts against the rest.

In [ ]:
repro = repro_raw.rename(columns={
    'Unnamed: 0': 'year', 'Unnamed: 3': 'Гадяцький', 'Unnamed: 6': 'Зіньківський',
    'Unnamed: 9': 'Золотоніський', 'Unnamed: 12': 'Кобеляцький', 'Unnamed: 15': 'Костянтиноградський',
    'Unnamed: 18': 'Кременчуцький', 'Unnamed: 21': 'Лохвицький', 'Unnamed: 24': 'Лубенський',
    'Unnamed: 27': 'Миргородський', 'Unnamed: 30': 'Переяславський', 'Unnamed: 33': 'Полтавський',
    'Unnamed: 36': 'Прилуцький', 'Unnamed: 39': 'Пирятинський', 'Unnamed: 42': 'Роменський',
    'Unnamed: 45': 'Хорольський', 'Unnamed: 48': 'Губернія загалом'})

repro = repro.drop(columns=['Губернія загалом'])
repro = repro.set_index(repro.columns[0])
repro = repro.loc[:, repro.iloc[0] == 'о.с.']      # keep the right metric columns
repro = repro.iloc[1:-2]                            # drop header / footer rows

long = repro.reset_index()
long = long.rename(columns={long.columns[0]: 'year'})
long = long.melt(id_vars=['year'], var_name='district_ua', value_name='reproduction')
long = long[long['year'] != 'Разом']               # drop the 'Total' row
long['reproduction'] = pd.to_numeric(long['reproduction'], errors='coerce')
long['district'] = long['district_ua'].map(DISTRICT_EN)
long.head()

In [ ]:
line = alt.Chart(long, title='Population reproduction over time').mark_line().encode(
    x=alt.X('year:O', title='Year'),
    y=alt.Y('reproduction:Q', title='Reproduction'),
    detail='district_ua:N',
    color=alt.condition(alt.FieldOneOfPredicate('district_ua', COSSACK_DISTRICTS),
                        alt.value('#c0392b'), alt.value('lightgray')),
    tooltip=['year:O', 'district:N', 'reproduction:Q']
).properties(width=720, height=380)
line

**Finding.** The top-Cossack districts (red) **track the overall provincial trend** - they rise and fall with the governorate as a whole, showing no distinct demographic behaviour.

## 5. Integrated dashboard
The four most representative views in one 2x2 layout.

In [ ]:
# Wide 2x2 layout, sized to fit a Notion cover frame nicely (landscape)
W, H, GAP = 560, 260, 40

dash_map     = map_share.properties(width=W, height=H, title='Cossack share per district')
dash_rank    = rank.properties(width=W, height=H, title='Districts ranked by Cossack share')
dash_scatter = land_scatter('land_state', 'state peasants').properties(
                   width=W, height=H, title='Cossack share vs land (state peasants)')
dash_line    = line.properties(width=W, height=H, title='Population reproduction over time')

dashboard = alt.vconcat(
    alt.hconcat(dash_map, dash_rank, spacing=GAP),
    alt.hconcat(dash_scatter, dash_line, spacing=GAP),
    spacing=30
).properties(
    title=alt.TitleParams(
        'Cossacks of the Poltava Governorate (1850) - Geography, Land & Demographics',
        subtitle='Distribution | concentration | land-per-capita | demographic dynamics',
        anchor='middle', fontSize=18)
).resolve_scale(color='independent').configure_view(stroke=None)

dashboard

In [ ]:
# Save an interactive copy for the portfolio
dashboard.save('poltava_census_dashboard.html')

## Conclusions

- **Distribution (Q1-Q2):** the largest Cossack counts are in Kobeliaky, Zolotonosha, Zinkiv and Romny; the highest Cossack *share* is in Zinkiv, Lokhvytsia and Lubny. These views describe the province and set up the analysis.
- **Land (Q3):** **no correlation** - a district's Cossack share does not explain land-per-capita for state or landlord peasants.
- **Demographics (Q4):** the top-Cossack districts **track the overall provincial trend** - no distinct demographic dynamic.

**Honest takeaway:** I expected "Cossack" districts to stand out and found that they don't. Cossack concentration simply wasn't a strong driver of land or demographic outcomes. Reporting that null result clearly - instead of forcing a narrative the data doesn't support - is part of doing the analysis properly.

## Data & tools
- **Source:** 1850 census of the Poltava Governorate (multi-sheet Excel) + a self-built GeoJSON of district borders.
- **Stack:** Python | pandas | GeoPandas | Altair / Vega-Lite.
- **AI:** used to help generate the GeoJSON name mapping and debug Altair syntax; the analysis and interpretation are my own.